# Data Preprocessing Pipeline: Индекс счастья

This notebook transforms raw JSON data into a structured and normalized format.

**Input:** Raw JSON dataset  
**Processing:** Type casting, column standardization, missing value handling, validation  
**Output:** Cleaned JSON dataset prepared for database upload

In [105]:
# Importing libraries
import pandas as pd
import matplotlib.pyplot as plt
import pycountry

In [107]:
# Helper function to transform country names to ISO3 (standard for all datasets in our project)
def to_iso3(name):
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None

In [109]:
df = pd.read_json('raw_data\happiness_rus_raw.json') # Check the directory where raw data file is stored

<>:1: SyntaxWarning: invalid escape sequence '\h'
<>:1: SyntaxWarning: invalid escape sequence '\h'
C:\Users\Honor\AppData\Local\Temp\ipykernel_36608\1713222713.py:1: SyntaxWarning: invalid escape sequence '\h'
  df = pd.read_json('raw_data\happiness_rus_raw.json') # Check the directory where raw data file is stored


In [111]:
print(df['year'].min())
print(df['year'].max())

2014
2024


In [113]:
# glancing at the data
df.head()

,year,country,score,rank
0,2014,Швейцария,7.587,1
1,2014,Исландия,7.561,2
2,2014,Дания,7.527,3
3,2014,Норвегия,7.522,4
4,2014,Канада,7.427,5


In [115]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1648 entries, 0 to 1647
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   year     1648 non-null   int64  
 1   country  1648 non-null   object 
 2   score    1648 non-null   float64
 3   rank     1648 non-null   int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 51.6+ KB


In [117]:
df.describe()

,year,score,rank
count,1648.000000,1648.000000,1648.000000
mean,2018.897451,5.459945,74.313107
std,3.160517,1.129218,42.593178
min,2014.000000,1.364000,1.000000
25%,2016.000000,4.601250,38.000000
50%,2019.000000,5.492000,74.000000
75%,2022.000000,6.309000,111.000000
max,2024.000000,7.842000,155.000000


In [119]:
# storing unique country names to translate in English (since all stored data is in English)
df['country'].unique()

array(['Швейцария', 'Исландия', 'Дания', 'Норвегия', 'Канада',
       'Финляндия', 'Нидерланды', 'Швеция', 'Новая Зеландия', 'Австралия',
       'Израиль', 'Коста-Рика', 'Австрия', 'Мексика', 'США', 'Бразилия',
       'Люксембург', 'Ирландия', 'Бельгия', 'ОАЭ', 'Великобритания',
       'Оман', 'Венесуэла', 'Сингапур', 'Панама', 'Германия', 'Чили',
       'Катар', 'Франция', 'Аргентина', 'Чехия', 'Уругвай', 'Колумбия',
       'Таиланд', 'Саудовская Аравия', 'Испания', 'Мальта', 'Тайвань',
       'Кувейт', 'Суринам', 'Тринидад и Тобаго', 'Сальвадор', 'Гватемала',
       'Узбекистан', 'Словакия', 'Япония', 'Южная Корея', 'Эквадор',
       'Бахрейн', 'Италия', 'Боливия', 'Молдова', 'Парагвай', 'Казахстан',
       'Словения', 'Литва', 'Никарагуа', 'Перу', 'Беларусь', 'Польша',
       'Малайзия', 'Хорватия', 'Ливия', 'Россия', 'Ямайка', 'Кипр',
       'Алжир', 'Косово', 'Туркменистан', 'Маврикий', 'Гонконг',
       'Эстония', 'Индонезия', 'Вьетнам', 'Турция', 'Киргизия', 'Нигерия',
       'Б

In [121]:
# I used LLM to map country names from Russian to English
russian_to_english = {
    'Финляндия': 'Finland','Дания': 'Denmark','Исландия': 'Iceland','Швеция': 'Sweden','Нидерланды': 'Netherlands','Коста-Рика': 'Costa Rica',
    'Норвегия': 'Norway','Израиль': 'Israel','Люксембург': 'Luxembourg','Мексика': 'Mexico','Австралия': 'Australia','Новая Зеландия': 'New Zealand',
    'Швейцария': 'Switzerland','Бельгия': 'Belgium','Ирландия': 'Ireland','Литва': 'Lithuania','Австрия': 'Austria','Канада': 'Canada',
    'Словения': 'Slovenia','Чехия': 'Czechia','ОАЭ': 'United Arab Emirates','Германия': 'Germany','Великобритания': 'United Kingdom','США': 'United States',
    'Белиз': 'Belize','Польша': 'Poland','Тайвань': 'Taiwan','Уругвай': 'Uruguay','Косово': 'Kosovo','Кувейт': 'Kuwait',
    'Сербия': 'Serbia','Саудовская Аравия': 'Saudi Arabia','Франция': 'France','Сингапур': 'Singapore','Румыния': 'Romania','Бразилия': 'Brazil',
    'Сальвадор': 'El Salvador','Испания': 'Spain','Эстония': 'Estonia','Италия': 'Italy','Панама': 'Panama','Аргентина': 'Argentina','Казахстан': 'Kazakhstan',
    'Гватемала': 'Guatemala','Чили': 'Chile','Вьетнам': 'Vietnam','Никарагуа': 'Nicaragua','Мальта': 'Malta','Таиланд': 'Thailand','Словакия': 'Slovakia','Латвия': 'Latvia',
    'Оман': 'Oman','Узбекистан': 'Uzbekistan','Парагвай': 'Paraguay','Япония': 'Japan','Босния и Герцеговина': 'Bosnia and Herzegovina','Филиппины': 'Philippines',
    'Южная Корея': 'South Korea','Бахрейн': 'Bahrain','Португалия': 'Portugal','Колумбия': 'Colombia','Эквадор': 'Ecuador','Гондурас': 'Honduras','Малайзия': 'Malaysia',
    'Перу': 'Peru','Россия': 'Russian Federation','Кипр': 'Cyprus','Китай': 'China','Венгрия': 'Hungary','Тринидад и Тобаго': 'Trinidad and Tobago',
    'Черногория': 'Montenegro','Хорватия': 'Croatia','Ямайка': 'Jamaica','Боливия': 'Bolivia','Киргизия': 'Kyrgyzstan',
    'Доминиканская Республика': 'Dominican Republic','Монголия': 'Mongolia',
    'Маврикий': 'Mauritius','Ливия': 'Libya','Молдова': 'Moldova','Греция': 'Greece','Венесуэла': 'Venezuela','Индонезия': 'Indonesia','Алжир': 'Algeria',
    'Болгария': 'Bulgaria','Северная Македония': 'North Macedonia','Армения': 'Armenia','Гонконг': 'Hong Kong','Албания': 'Albania','Таджикистан': 'Tajikistan',
    'Грузия': 'Georgia','Непал': 'Nepal','Лаос': 'Laos','Турция': 'Turkey','ЮАР': 'South Africa','Мозамбик': 'Mozambique','Габон': 'Gabon',
    'Кот-д’Ивуар': "Côte d'Ivoire",'Иран': 'Iran','Конго': 'Congo','Ирак': 'Iraq','Гвинея': 'Guinea',
    'Намибия': 'Namibia','Камерун': 'Cameroon','Нигерия': 'Nigeria','Азербайджан': 'Azerbaijan','Сенегал': 'Senegal',
    'Палестинские территории': 'Palestine, State of','Пакистан': 'Pakistan','Нигер': 'Niger','Украина': 'Ukraine','Марокко': 'Morocco','Тунис': 'Tunisia',
    'Мавритания': 'Mauritania','Кения': 'Kenya','Уганда': 'Uganda','Гамбия': 'Gambia','Индия': 'India','Чад': 'Chad','Буркина-Фасо': 'Burkina Faso','Бенин': 'Benin',
    'Сомали': 'Somalia','Мали': 'Mali','Камбоджа': 'Cambodia','Гана': 'Ghana','Мьянма': 'Myanmar','Того': 'Togo','Иордания': 'Jordan','Либерия': 'Liberia',
    'Мадагаскар': 'Madagascar','Замбия': 'Zambia','Эфиопия': 'Ethiopia','Шри-Ланка': 'Sri Lanka','Бангладеш': 'Bangladesh','Египет': 'Egypt','Танзания': 'Tanzania','Эсватини': 'Eswatini',
    'Лесото': 'Lesotho','Коморские острова': 'Comoros','Йемен': 'Yemen','Конго ДР': 'DR Congo','Ботсвана': 'Botswana',
    'Зимбабве': 'Zimbabwe','Малави': 'Malawi','Ливан': 'Lebanon','Сьерра-Леоне': 'Sierra Leone','Афганистан': 'Afghanistan', 
    'Сирия': 'Syrian Arab Republic', 'Катар': 'Qatar', 'Суринам': 'Suriname', 'Беларусь' : 'Belarus', 'Туркменистан' : 'Turkmenistan', 
    'Бутан' : 'Bhutan', 'Судан' : 'Sudan', 'Гаити' : 'Haiti', 'Джибути': 'Djibouti', 'Ангола' : 'Angola', 'ЦАР' : 'Central African Republic', 
    'Руанда' : 'Rwanda', 'Бурунди': 'Burundi', 'Пуэрто-Рико' : 'Puerto Rico', 'Южный Судан': 'South Sudan', 'Мальдивские острова': 'Maldives'
}

In [123]:
df['country_english'] = df['country'].map(russian_to_english) # mapping the data

In [125]:
# removing names in russian
df = df.drop(columns=["country"])

In [127]:
df['country_code'] = df['country_english'].apply(to_iso3) # adding a column with ISO3 code

In [129]:
# detecting country names for which the function didn't return their ISO3 code
df[df["country_code"].isna()]['country_english'].unique()

array(['Kosovo', 'Turkey', 'DR Congo'], dtype=object)

In [131]:
# Since not all country_names were mapped, I manually change them
manual_map = {
    "Turkey": "TUR",
    "Kosovo": "XKX",
    "DR Congo" : "COD"
}

df["country_code"] = df["country_code"].fillna(df["country_english"].map(manual_map))

In [133]:
df[df["country_code"].isna()]

,year,score,rank,country_english,country_code


In [135]:
df['country_year'] = df["country_code"] + df["year"].astype(str)

In [137]:
df.rename(columns={"country_english": "country"}, inplace = True)

In [139]:
df.head() # long format of dataset is ready

,year,score,rank,country,country_code,country_year
0,2014,7.587,1,Switzerland,CHE,CHE2014
1,2014,7.561,2,Iceland,ISL,ISL2014
2,2014,7.527,3,Denmark,DNK,DNK2014
3,2014,7.522,4,Norway,NOR,NOR2014
4,2014,7.427,5,Canada,CAN,CAN2014


In [141]:
# pivoting the table so that only unique country names are in the rows
df_wide = df.pivot(index = 'country', columns = 'year', values = 'score')

In [143]:
df_wide.reset_index(inplace=True)

In [145]:
df_wide['country_code'] = df_wide['country'].apply(to_iso3) # adding a column with ISO3 code

In [147]:
# detecting country names for which the function didn't return their ISO3 code
df_wide[df_wide["country_code"].isna()]['country'].unique()

array(['DR Congo', 'Kosovo', 'Turkey'], dtype=object)

In [151]:
# Since not all country_names were mapped, I manually change them
df_wide["country_code"] = df_wide["country_code"].fillna(df_wide["country"].map(manual_map))

In [153]:
df_wide.head()

year,country,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,country_code
0,Afghanistan,3.575,3.360,3.794,3.632,3.203,2.567,2.523,2.404,1.859,1.721,1.364,AFG
1,Albania,4.959,4.655,4.644,4.586,4.719,4.883,5.117,5.199,5.277,5.304,5.411,ALB
2,Algeria,5.605,6.355,5.872,5.295,5.211,5.005,4.887,5.122,5.329,5.364,5.571,DZA
3,Angola,4.033,3.866,3.795,3.795,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AGO
4,Argentina,6.574,6.650,6.599,6.388,6.086,5.975,5.929,5.967,6.024,6.188,6.397,ARG


## Saving preprocessed datasets

In [155]:
df.to_json(
    "processed_data/hpi_russian_clean.json",
    orient="records",
    indent=2,
    force_ascii=False
)

In [157]:
df_wide.to_json(
    "processed_data/hpi_russian_wide_clean.json",
    orient="records",
    indent=2,
    force_ascii=False
)